# Chapter 8: Intro to Regularization — Learning Signal and Ignoring Noise
### Grokking Deep Learning (Andrew W. Trask)
**Student Name**: Muhammad Aidil Fadhli Awalludin  
**NIM**: 101032300087  
**Class**: TK4701  

---

> *"Entities should not be multiplied beyond necessity."*  
> — Occam's Razor

---

## 1. Executive Summary & Objective

In deep learning, raw optimization capacity is a double-edged sword. A large neural network has sufficient parameter degrees of freedom to memorize the training data verbatim—including idiosyncratic noise—leading to **Overfitting**.

### Core Objectives:
1. **Analyze Overfitting and the Bias-Variance Tradeoff**: Understand the divergence between training error and generalization (testing) error.
2. **Dissect Regularization Mechanisms**: Study Early Stopping, L2 Regularization (Weight Decay), and Dropout.
3. **Implement Dropout from First Principles**: Construct random binary masking during forward propagation and route gradients strictly through surviving neurons during backpropagation.
4. **Empirical Verification**: Demonstrate how Dropout prevents neuron co-adaptation, significantly improving test accuracy.

---
## 2. Theoretical Deep-Dive

### 2.1 Signal vs. Noise and Overfitting

Real-world data consists of two components:
$$\text{Observed Data} = \text{Underlying Signal} + \text{Random Noise}$$

- **Signal**: The invariant, generalizable relationship governing the phenomenon (e.g., circular shapes indicate digits '0' or '8').
- **Noise**: Dataset-specific anomalies and measurement artifacts (e.g., an accidental smudge on an image).

```
Error
  ^                     
  |        \               /  <-- Testing Error (Overfitting regime)
  |         \             /   
  |          \___________/    <-- Optimal Generalization Point (Early Stopping)
  |           \         
  |            \______________ <-- Training Error
  +-------------------------------------> Epochs / Model Capacity
```

When a network overfits, it achieves near-zero training error by memorizing noise, but its error on unseen test data escalates sharply.

### 2.2 Dropout: The Ultimate Ensemble Method

Proposed by Nitish Srivastava, Geoffrey Hinton, et al. (2014), **Dropout** is an exceptionally effective regularizer.

#### The Mechanism:
During each training iteration, every hidden neuron is independently deactivated ("dropped") with probability $p$ (commonly $p = 0.5$):

$$\mathbf{m} \sim \text{Bernoulli}(1 - p)$$
$$\mathbf{a}_{dropout} = \mathbf{a} \odot \mathbf{m}$$

To ensure the expected sum of activations remains consistent between training (when half the neurons are dead) and testing (when all neurons are active), we scale surviving activations by $\frac{1}{1 - p}$ (known as **Inverted Dropout**):

$$\mathbf{a}_{dropout} = \frac{\mathbf{a} \odot \mathbf{m}}{1 - p}$$

#### Why Dropout Prevents Overfitting:
1. **Prevents Co-Adaptation**: Neurons cannot rely on specific neighboring nodes to fix their mistakes. Every neuron is forced to learn useful, self-reliant representations.
2. **Implicit Ensembling**: Dropping neurons samples from $2^N$ possible sub-network topologies. At test time, using all weights approximates taking the geometric mean over this massive ensemble of sub-networks.

---
## 3. Code Reproduction

We now implement Dropout and evaluate its impact against an unregularized baseline on a noisy classification task.

In [ ]:
# 3.1 Data Preparation with Artificial Noise Injection
import numpy as np

np.random.seed(1)

# 500 samples, 40 features, 2 classes
N = 500
D = 40
X = np.random.randn(N, D)
# True signal depends only on first 3 features; remaining 37 features are pure random noise
true_signal = X[:, 0] * 1.5 - X[:, 1] * 2.0 + X[:, 2] * 0.8
y = (true_signal > 0).astype(float).reshape(-1, 1)

# Split: 100 Train (high risk of overfitting), 400 Test
X_train, X_test = X[:100], X[100:]
y_train, y_test = y[:100], y[100:]

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

In [ ]:
# 3.2 Experiment A: Unregularized Baseline Network (Prone to Overfitting)

def relu(x):
    return (x > 0) * x

def relu2deriv(output):
    return output > 0

hidden_size = 64
alpha = 0.05
epochs = 300

np.random.seed(1)
w_0_1 = 0.2 * np.random.random((D, hidden_size)) - 0.1
w_1_2 = 0.2 * np.random.random((hidden_size, 1)) - 0.1

for epoch in range(epochs):
    for i in range(len(X_train)):
        l0 = X_train[i:i+1]
        l1 = relu(np.dot(l0, w_0_1))
        l2 = np.dot(l1, w_1_2)
        
        l2_delta = l2 - y_train[i:i+1]
        l1_delta = l2_delta.dot(w_1_2.T) * relu2deriv(l1)
        
        w_1_2 -= alpha * l1.T.dot(l2_delta)
        w_0_1 -= alpha * l0.T.dot(l1_delta)

# Evaluate baseline
train_preds = (np.dot(relu(np.dot(X_train, w_0_1)), w_1_2) > 0.5).astype(float)
test_preds = (np.dot(relu(np.dot(X_test, w_0_1)), w_1_2) > 0.5).astype(float)

acc_train_base = np.mean(train_preds == y_train) * 100
acc_test_base = np.mean(test_preds == y_test) * 100

print("Baseline (No Dropout):")
print(f"  Train Accuracy: {acc_train_base:.2f}% (Memorized noisy features)")
print(f"  Test Accuracy:  {acc_test_base:.2f}%")

In [ ]:
# 3.3 Experiment B: Network Trained with Dropout (p = 0.5)

np.random.seed(1)
w_0_1_drop = 0.2 * np.random.random((D, hidden_size)) - 0.1
w_1_2_drop = 0.2 * np.random.random((hidden_size, 1)) - 0.1

for epoch in range(epochs):
    for i in range(len(X_train)):
        l0 = X_train[i:i+1]
        l1 = relu(np.dot(l0, w_0_1_drop))
        
        # Inverted Dropout Mask: random binary 0 or 1, scaled by 2.0 (for p = 0.5)
        dropout_mask = np.random.randint(2, size=l1.shape)
        l1 *= dropout_mask * 2
        
        l2 = np.dot(l1, w_1_2_drop)
        
        l2_delta = l2 - y_train[i:i+1]
        # Pass error delta strictly through non-dropped neurons
        l1_delta = l2_delta.dot(w_1_2_drop.T) * relu2deriv(l1)
        l1_delta *= dropout_mask
        
        w_1_2_drop -= alpha * l1.T.dot(l2_delta)
        w_0_1_drop -= alpha * l0.T.dot(l1_delta)

# Test phase: evaluate with ALL neurons active (no dropout mask applied)
train_preds_drop = (np.dot(relu(np.dot(X_train, w_0_1_drop)), w_1_2_drop) > 0.5).astype(float)
test_preds_drop = (np.dot(relu(np.dot(X_test, w_0_1_drop)), w_1_2_drop) > 0.5).astype(float)

acc_train_drop = np.mean(train_preds_drop == y_train) * 100
acc_test_drop = np.mean(test_preds_drop == y_test) * 100

print("Regularized Model (With Dropout):")
print(f"  Train Accuracy: {acc_train_drop:.2f}%")
print(f"  Test Accuracy:  {acc_test_drop:.2f}%")
print(f"\nGeneralization Improvement: +{acc_test_drop - acc_test_base:.2f}% on unseen test data!")

---
## 4. Chapter 8 Summary & Key Takeaways

| Regularization Technique | Mathematical / Algorithmic Form | How It Prevents Overfitting |
|:---|:---:|:---|
| **Early Stopping** | $\text{stop if } \text{Val Loss} \uparrow$ | Prevents the model from entering the late-stage noise memorization regime. |
| **L2 Weight Decay** | $L + \frac{1}{2}\lambda \|\mathbf{W}\|^2$ | Encourages smaller, evenly distributed weights; penalizes over-reliance on individual features. |
| **Dropout** | $\mathbf{a} \odot \mathbf{m} \cdot \frac{1}{1-p}$ | Breaks neuron co-adaptation by randomly severing hidden nodes; simulates massive sub-network ensembling. |
| **Next Step** | In **Chapter 9**, we move beyond squared error and linear outputs to examine **Activation Functions** that model true probabilities (Sigmoid, Softmax, Cross-Entropy). |